<div style="background:#2C3E7A;padding:30px;border-radius:10px;color:white;text-align:center">
<h1 style="color:white;margin:0">🏦 GreenAlpha Challenge</h1>
<h3 style="color:#27AE60;margin:8px 0">Serious Game — Gestion de Portefeuille & Finance Durable</h3>
<p style="color:#ccc;margin:0">M1 Finance — Grenoble IAE — Investissements et Marchés Financiers</p>
</div>

---

### 🎯 Votre mission

Vous êtes **gérants juniors** chez *GreenAlpha Asset Management*. En **2 heures**, votre équipe doit construire, défendre et optimiser un portefeuille d'actions européennes — d'abord librement, puis sous contrainte ESG imposée par une nouvelle directive réglementaire.

**L'équipe avec le meilleur score remporte le trophée GreenAlpha 🏆**

| Phase | Durée | Contenu |
|---|---|---|
| ⚡ Briefing | 15 min | Règles, formation des équipes, découverte des actifs |
| 🎯 Acte 1 | 25 min | Construire un portefeuille libre (Markowitz) |
| 📰 Événement | 5 min | Choc de marché révélé par le professeur |
| 🌿 Acte 2 | 25 min | Recomposer sous contrainte ESG (directive SFDR) |
| 🎤 Pitch | 20 min | Défendre vos choix devant le groupe |
| 📊 Correction | 10 min | Révélation du portefeuille optimal + synthèse |

---

## ⚡ Setup — Exécutez cette cellule en premier

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from ipywidgets import interact, IntSlider, FloatSlider
import random, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#FAFAFA',
    'axes.grid':True,'grid.alpha':0.25,'font.family':'sans-serif',
    'axes.spines.top':False,'axes.spines.right':False
})
C_BLUE='#2C3E7A'; C_GREEN='#27AE60'; C_RED='#E74C3C'; C_GOLD='#F39C12'

# ── Univers d'actifs ─────────────────────────────────────────────────────
ASSETS = {
    'TotalEnergies': {'ret':0.082,'vol':0.221,'esg':32,'sector':'Énergie'},
    'Schneider':     {'ret':0.134,'vol':0.198,'esg':78,'sector':'Industrie'},
    'Air Liquide':   {'ret':0.096,'vol':0.172,'esg':65,'sector':'Chimie'},
    'BNP Paribas':   {'ret':0.071,'vol':0.263,'esg':44,'sector':'Finance'},
    'Danone':        {'ret':0.058,'vol':0.189,'esg':71,'sector':'Conso.'},
    'Stellantis':    {'ret':0.105,'vol':0.298,'esg':29,'sector':'Auto'},
    "L'Oréal":       {'ret':0.118,'vol':0.181,'esg':82,'sector':'Luxe'},
    'Vinci':         {'ret':0.089,'vol':0.207,'esg':56,'sector':'Infra'},
    'Sanofi':        {'ret':0.076,'vol':0.194,'esg':68,'sector':'Santé'},
    'Engie':         {'ret':0.063,'vol':0.241,'esg':51,'sector':'Utilities'},
}
NAMES = list(ASSETS.keys()); N = len(NAMES)
RETS  = np.array([ASSETS[a]['ret'] for a in NAMES])
VOLS  = np.array([ASSETS[a]['vol'] for a in NAMES])
ESG   = np.array([ASSETS[a]['esg'] for a in NAMES])

# Matrice de covariance
np.random.seed(42)
_A = np.random.randn(N,N)*0.15; _C = np.eye(N)+_A+_A.T
_C /= np.abs(_C).max(axis=1,keepdims=True); np.fill_diagonal(_C,1.0)
_C = (_C+_C.T)/2
_ev = np.linalg.eigvals(_C)
if _ev.min()<0: _C+=(-_ev.min()+0.01)*np.eye(N)
_C /= np.sqrt(np.outer(np.diag(_C),np.diag(_C))); np.fill_diagonal(_C,1.0)
COV = np.outer(VOLS,VOLS)*_C

# ── Fonctions d'optimisation ─────────────────────────────────────────────
def port_stats(w, rets=None):
    if rets is None: rets = RETS
    return w@rets, np.sqrt(w@COV@w)

def optimize(esg_min=None, rf=0.025, rets=None):
    if rets is None: rets = RETS
    cons = [{'type':'eq','fun':lambda w: np.sum(w)-1}]
    if esg_min: cons.append({'type':'ineq','fun':lambda w,s=esg_min: w@ESG-s})
    def neg_sharpe(w):
        r,v = port_stats(w, rets)
        return -(r-rf)/v if v>1e-8 else 0
    res = minimize(neg_sharpe, x0=np.ones(N)/N, bounds=[(0,1)]*N,
                   constraints=cons, method='SLSQP',
                   options={'ftol':1e-10,'maxiter':2000})
    return res.x if res.success else np.ones(N)/N

def frontier(esg_min=None, n_pts=70, rets=None):
    if rets is None: rets = RETS
    fv, fr = [], []
    for tr in np.linspace(rets.min()+0.002, rets.max()-0.002, n_pts):
        cons = [{'type':'eq','fun':lambda w: np.sum(w)-1},
                {'type':'eq','fun':lambda w,r=tr: w@rets-r}]
        if esg_min: cons.append({'type':'ineq','fun':lambda w,s=esg_min: w@ESG-s})
        res = minimize(lambda w: w@COV@w, x0=np.ones(N)/N, bounds=[(0,1)]*N,
                       constraints=cons, method='SLSQP',
                       options={'ftol':1e-10,'maxiter':2000})
        if res.success:
            r,v = port_stats(res.x, rets)
            fv.append(v); fr.append(r)
    return np.array(fv), np.array(fr)

def compute_score(w, rf=0.025, rets=None, bonus=0, label=''):
    if rets is None: rets = RETS
    r,v = port_stats(w, rets)
    sh  = (r-rf)/v if v>1e-8 else 0
    esg = w@ESG
    s_sharpe = min(max(sh/1.5,0),1)*50
    s_esg    = min(esg/100,1)*30
    total    = round(s_sharpe+s_esg+bonus,1)
    print(f'\n  📊 Score {label}')
    print(f'  {'─'*38}')
    print(f'  Sharpe   : {sh:.3f}  → {s_sharpe:.1f}/50 pts')
    print(f'  ESG      : {esg:.1f}/100  → {s_esg:.1f}/30 pts')
    if bonus: print(f'  Bonus    : +{bonus:.0f} pts 🎁')
    print(f'  {'─'*38}')
    print(f'  TOTAL    : {total}/80 pts')
    return total, sh, esg

print('✅ Setup complet — prêt à jouer !')

---
# ⚡ BRIEFING — Découvrez votre univers d'investissement

In [ ]:
# Tableau des actifs
df_assets = pd.DataFrame({
    'Actif':          NAMES,
    'Secteur':        [ASSETS[a]['sector'] for a in NAMES],
    'Rendement (%)':  (RETS*100).round(1),
    'Volatilité (%)': (VOLS*100).round(1),
    'Score ESG /100': ESG,
})

def color_esg(val):
    if val>=65: return 'background-color:#d4edda;color:#155724;font-weight:bold'
    elif val>=45: return 'background-color:#fff3cd;color:#856404'
    return 'background-color:#f8d7da;color:#721c24'

display(df_assets.style
    .applymap(color_esg, subset=['Score ESG /100'])
    .set_caption('🔍 Votre univers — Analysez avant de décider !')
    .set_table_styles([{'selector':'caption','props':[('font-size','14px'),('font-weight','bold')]}])
)

# Carte risque/rendement
fig, ax = plt.subplots(figsize=(10,6))
sc = ax.scatter(VOLS*100, RETS*100, c=ESG, cmap='RdYlGn', vmin=0, vmax=100,
                s=160, zorder=5, edgecolors='white', lw=1.5)
plt.colorbar(sc, ax=ax, label='Score ESG')
for i,name in enumerate(NAMES):
    ax.annotate(name,(VOLS[i]*100,RETS[i]*100),
                textcoords='offset points',xytext=(7,4),fontsize=9)
ax.axhline(np.mean(RETS)*100, color='gray', ls=':', alpha=0.5, label='Rend. moyen')
ax.axvline(np.mean(VOLS)*100, color='gray', ls='--', alpha=0.5, label='Vol. moyenne')
ax.set_xlabel('Volatilité (%)'); ax.set_ylabel('Rendement espéré (%)')
ax.set_title('Espace Risque / Rendement — Actifs disponibles', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
# 🎯 ACTE 1 — Construisez votre portefeuille libre (25 min)

> **Consigne :** Vous n'avez **aucune contrainte** pour l'instant.  
> Objectif : maximiser le ratio de Sharpe (performance ajustée du risque).  
> Taux sans risque = **2.5%** (OAT 10 ans)

### Étape A — Explorez la frontière efficiente

In [ ]:
# Calcul frontière classique + portefeuille optimal de référence
RF = 0.025
fv_ref, fr_ref = frontier(esg_min=None)
w_opt_ref      = optimize(esg_min=None, rf=RF)
r_ref, v_ref   = port_stats(w_opt_ref)

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(VOLS*100, RETS*100, c=ESG, cmap='RdYlGn', vmin=0, vmax=100,
           s=120, zorder=5, edgecolors='white', lw=1.2)
for i,name in enumerate(NAMES):
    ax.annotate(name,(VOLS[i]*100,RETS[i]*100),
                textcoords='offset points',xytext=(6,3),fontsize=8.5)
ax.plot(fv_ref*100, fr_ref*100, '--', color=C_BLUE, lw=2.5, label='Frontière efficiente')
ax.scatter([v_ref*100],[r_ref*100], marker='*', s=300, color=C_GOLD, zorder=6,
           label=f'Max Sharpe = {(r_ref-RF)/v_ref:.3f} ⭐')
# Ligne de marché des capitaux
x_cml = np.linspace(0, max(VOLS)*100*1.1, 100)
ax.plot(x_cml, RF*100+(r_ref-RF)/v_ref*x_cml, ':', color=C_GOLD, alpha=0.6, label='CML')
ax.scatter([0],[RF*100], marker='o', s=80, color='gray', zorder=6, label=f'Rf={RF*100:.1f}%')
ax.set_xlabel('Volatilité (%)'); ax.set_ylabel('Rendement espéré (%)')
ax.set_title('Frontière efficiente — Acte 1', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
print(f'\n💡 Portefeuille Max Sharpe de référence : Rend={r_ref*100:.2f}% | Vol={v_ref*100:.2f}% | Sharpe={(r_ref-RF)/v_ref:.3f} | ESG={w_opt_ref@ESG:.1f}')

### Étape B — 💬 Discussion d'équipe (10 min)

Avant de saisir vos poids, discutez :
1. Quels actifs sont dans le quadrant **haut-gauche** (bon rendement, faible risque) ?
2. Vaut-il mieux concentrer ou diversifier ?
3. Est-ce que le score ESG influence votre choix à ce stade ?

### Étape C — ✏️ Saisissez vos poids (ils doivent sommer à 100%)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ✏️  VOTRE ÉQUIPE : saisissez vos poids (somme = 100)
# ══════════════════════════════════════════════════════════════
NOM_EQUIPE = "MonEquipe"    # <-- MODIFIER

poids_acte1 = {
    'TotalEnergies': 0,
    'Schneider':     0,
    'Air Liquide':   0,
    'BNP Paribas':   0,
    'Danone':        0,
    'Stellantis':    0,
    "L'Oréal":       0,
    'Vinci':         0,
    'Sanofi':        0,
    'Engie':         0,
}
# ══════════════════════════════════════════════════════════════

w1 = np.array([poids_acte1[n] for n in NAMES], dtype=float) / 100
total = sum(poids_acte1.values())

if abs(total - 100) > 0.1:
    print(f'⚠️  Somme = {total}% — elle doit être 100%. Corrigez !')
else:
    print(f'✅ Portefeuille Acte 1 — {NOM_EQUIPE}')
    r1,v1 = port_stats(w1)
    print(f'   Rendement : {r1*100:.2f}% | Volatilité : {v1*100:.2f}% | '
          f'Sharpe : {(r1-RF)/v1:.3f} | ESG moyen : {w1@ESG:.1f}/100')
    df_w1 = pd.DataFrame({'Actif':NAMES,'Poids (%)':list(poids_acte1.values()),'ESG':ESG})
    display(df_w1[df_w1['Poids (%)']>0].style.applymap(color_esg, subset=['ESG']))

---
# 📰 ÉVÉNEMENT — Choc de marché !

> **⏳ Attendez que le professeur révèle l'événement avant d'exécuter cette cellule.**

In [ ]:
# ══════════════════════════════════════════════════════════════
#  🎲 PROFESSEUR : choisissez l'événement (1, 2 ou 3)
#  1 = Krach sectoriel Énergie (-15%)
#  2 = Scandale ESG Stellantis (-20% + malus ESG)
#  3 = Rally Green Tech : Schneider & Air Liquide (+15%)
EVENEMENT = 1   # <-- MODIFIER (professeur)
# ══════════════════════════════════════════════════════════════

RETS_CHOC = RETS.copy()
ESG_CHOC  = ESG.copy()
desc_evt  = ''
bonus_evt = 0

if EVENEMENT == 1:
    desc_evt = '📉 KRACH SECTORIEL ÉNERGIE : TotalEnergies chute de 15%'
    RETS_CHOC[NAMES.index('TotalEnergies')] -= 0.15
elif EVENEMENT == 2:
    desc_evt = '💥 SCANDALE ESG STELLANTIS : fraude aux émissions détectée (-20% + ESG effondré)'
    RETS_CHOC[NAMES.index('Stellantis')] -= 0.20
    ESG_CHOC[NAMES.index('Stellantis')]   = 5
elif EVENEMENT == 3:
    desc_evt = '🌿 RALLY GREEN TECH : Schneider +15%, Air Liquide +10% (accélération transition)'
    RETS_CHOC[NAMES.index('Schneider')]   += 0.15
    RETS_CHOC[NAMES.index('Air Liquide')] += 0.10

print(f'\n{'═'*60}')
print(f'  📰 ÉVÉNEMENT : {desc_evt}')
print(f'{'═'*60}')
print()

# Impact sur le portefeuille Acte 1
if sum(poids_acte1.values()) == 100:
    r1_choc, v1_choc = port_stats(w1, RETS_CHOC)
    print(f'  Impact sur votre portefeuille Acte 1 :')
    print(f'  Rendement : {RETS@w1*100:.2f}% → {r1_choc*100:.2f}% ({(r1_choc-RETS@w1)*100:+.2f}pp)')
    print(f'  Sharpe    : {(RETS@w1-RF)/np.sqrt(w1@COV@w1):.3f} → {(r1_choc-RF)/v1_choc:.3f}')

---
# 🌿 ACTE 2 — Recomposez sous contrainte ESG (25 min)

> **Contexte réglementaire :** Suite à l'événement, la Commission Européenne impose une mise à jour de la directive SFDR.  
> Votre fonds doit désormais afficher un **score ESG moyen ≥ 55** pour conserver son label Article 8.  
> Sans ce label, vous perdez 30% de votre encours sous gestion. **Vous devez vous conformer.**

### Étape A — Explorez la nouvelle frontière

In [ ]:
ESG_MIN = 55  # seuil SFDR imposé

# Frontières avec rendements choqués
fv_c2, fr_c2 = frontier(esg_min=None,    rets=RETS_CHOC)
fv_e2, fr_e2 = frontier(esg_min=ESG_MIN, rets=RETS_CHOC)
w_opt_c2     = optimize(esg_min=None,    rf=RF, rets=RETS_CHOC)
w_opt_e2     = optimize(esg_min=ESG_MIN, rf=RF, rets=RETS_CHOC)
r_c2,v_c2    = port_stats(w_opt_c2, RETS_CHOC)
r_e2,v_e2    = port_stats(w_opt_e2, RETS_CHOC)

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# Panneau gauche : frontières
ax = axes[0]
ax.scatter(VOLS*100, RETS_CHOC*100, c=ESG_CHOC, cmap='RdYlGn', vmin=0, vmax=100,
           s=120, zorder=5, edgecolors='white', lw=1.2)
for i,name in enumerate(NAMES):
    ax.annotate(name,(VOLS[i]*100,RETS_CHOC[i]*100),
                textcoords='offset points',xytext=(6,3),fontsize=8.5)
ax.plot(fv_c2*100,fr_c2*100,'--',color=C_BLUE,lw=2,label='Frontière libre')
ax.plot(fv_e2*100,fr_e2*100,'-', color=C_GREEN,lw=2.5,label=f'Frontière ESG≥{ESG_MIN}')
ax.scatter([v_c2*100],[r_c2*100],marker='*',s=280,color=C_BLUE,zorder=6,
           label=f'Max Sharpe libre ({(r_c2-RF)/v_c2:.3f})')
ax.scatter([v_e2*100],[r_e2*100],marker='*',s=280,color=C_GREEN,zorder=6,
           label=f'Max Sharpe ESG ({(r_e2-RF)/v_e2:.3f})')
# Annotation coût ESG
ax.annotate('', xy=(v_e2*100,r_e2*100), xytext=(v_c2*100,r_c2*100),
            arrowprops=dict(arrowstyle='<->',color='orange',lw=2))
mid_x = (v_c2+v_e2)*50
mid_y = (r_c2+r_e2)*50
ax.text(mid_x+0.2, mid_y, 'coût\nESG', color='orange', fontsize=9, va='center')
ax.set_xlabel('Volatilité (%)'); ax.set_ylabel('Rendement (%)')
ax.set_title('Frontières après événement', fontweight='bold')
ax.legend(fontsize=8.5)

# Panneau droit : comparaison barres
ax2 = axes[1]
cats  = ['Sharpe','Rend. (%)','Vol. (%)','ESG moyen']
v_c   = [(r_c2-RF)/v_c2, r_c2*100, v_c2*100, w_opt_c2@ESG_CHOC]
v_e   = [(r_e2-RF)/v_e2, r_e2*100, v_e2*100, w_opt_e2@ESG_CHOC]
x     = np.arange(len(cats)); w_bar = 0.35
b1    = ax2.bar(x-w_bar/2, v_c, w_bar, label='Libre',    color=C_BLUE,  alpha=0.85)
b2    = ax2.bar(x+w_bar/2, v_e, w_bar, label=f'ESG≥{ESG_MIN}', color=C_GREEN, alpha=0.85)
for b in list(b1)+list(b2):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.15,
             f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=9)
cout = ((r_c2-RF)/v_c2-(r_e2-RF)/v_e2)/((r_c2-RF)/v_c2)*100
ax2.set_title(f'Comparaison optimaux\nCoût ESG sur Sharpe : {cout:.1f}%', fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(cats); ax2.legend()
ax2.set_ylim(0, max(max(v_c),max(v_e))*1.22)

plt.suptitle('Acte 2 — Impact de la contrainte ESG', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print(f'\n💡 Coût ESG sur le Sharpe : {cout:.1f}%  —  Bon ou mauvais marché pour la planète ?')

### Étape B — 💬 Rediscutez en équipe (8 min)

1. Faut-il juste dépasser ESG 55, ou aller plus loin ?
2. Quels actifs devez-vous **renforcer** ? Lesquels **réduire** ?
3. Comment l'événement change-t-il vos convictions ?

### Étape C — ✏️ Votre portefeuille Acte 2

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ✏️  Nouveaux poids — contrainte ESG ≥ 55 obligatoire
poids_acte2 = {
    'TotalEnergies': 0,
    'Schneider':     0,
    'Air Liquide':   0,
    'BNP Paribas':   0,
    'Danone':        0,
    'Stellantis':    0,
    "L'Oréal":       0,
    'Vinci':         0,
    'Sanofi':        0,
    'Engie':         0,
}
# ══════════════════════════════════════════════════════════════

w2    = np.array([poids_acte2[n] for n in NAMES], dtype=float) / 100
total2 = sum(poids_acte2.values())
esg2   = w2 @ ESG_CHOC

if abs(total2-100) > 0.1:
    print(f'⚠️  Somme = {total2}% — doit être 100%')
elif esg2 < ESG_MIN:
    print(f'🚫 Score ESG = {esg2:.1f} < {ESG_MIN} — contrainte SFDR NON respectée !')
    print(f'   Vous perdez le label Art.8 — pénalité de 15 pts.')
else:
    print(f'✅ Contrainte ESG respectée : {esg2:.1f} ≥ {ESG_MIN}')
    r2,v2 = port_stats(w2, RETS_CHOC)
    print(f'   Rendement : {r2*100:.2f}% | Volatilité : {v2*100:.2f}% | '
          f'Sharpe : {(r2-RF)/v2:.3f} | ESG : {esg2:.1f}')
    df_w2 = pd.DataFrame({'Actif':NAMES,'Poids (%)':list(poids_acte2.values()),'ESG':ESG_CHOC})
    display(df_w2[df_w2['Poids (%)']>0].style.applymap(color_esg, subset=['ESG']))

---
# 📊 SCORE FINAL

In [ ]:
assert abs(sum(poids_acte1.values())-100)<0.1, 'Acte 1 incomplet'
assert abs(sum(poids_acte2.values())-100)<0.1, 'Acte 2 incomplet'

print(f'\n  🏦 {NOM_EQUIPE} — Score Final')
print(f'  {"═"*50}')

# Score Acte 1 (rendements normaux)
s1, sh1, esg1 = compute_score(w1, rf=RF, rets=RETS, label='Acte 1 (avant événement)')

# Score Acte 2 — pénalité si ESG < 55
esg2_val = w2 @ ESG_CHOC
penalite = -15 if esg2_val < ESG_MIN else 0
bonus_conformite = 10 if esg2_val >= 65 else 0  # bonus dépassement
s2, sh2, esg2_score = compute_score(w2, rf=RF, rets=RETS_CHOC,
                                     bonus=penalite+bonus_conformite,
                                     label='Acte 2 (après événement + ESG)')
if penalite: print('  ⚠️  Pénalité -15 pts : contrainte ESG non respectée')
if bonus_conformite: print('  🌿 Bonus +10 pts : ESG exemplaire (≥65) !')

score_total = round(s1 + s2, 1)
print(f'\n  {"═"*50}')
print(f'  🏆 SCORE TOTAL : {s1:.1f} + {s2:.1f} = {score_total:.1f} / 160 pts')
print(f'  {"═"*50}')

# Barre de progression visuelle
pct = score_total / 160
bar = '█' * int(pct*30) + '░' * (30-int(pct*30))
print(f'  [{bar}] {pct*100:.0f}%')

---
# 🎤 PITCH — Défendez vos choix (3 min/équipe)

Préparez votre argumentaire sur les questions suivantes :

In [ ]:
# Visualisation comparative Acte 1 vs Acte 2
fig, axes = plt.subplots(1, 2, figsize=(13,5))

for ax, w, titre, rets_used, esg_used in [
    (axes[0], w1, f'Acte 1 — {NOM_EQUIPE}', RETS, ESG),
    (axes[1], w2, f'Acte 2 — {NOM_EQUIPE} (post-événement)', RETS_CHOC, ESG_CHOC)
]:
    idx  = np.where(w > 0.005)[0]
    lbls = [NAMES[i] for i in idx]
    vals = [w[i]*100 for i in idx]
    esgs = [esg_used[i] for i in idx]
    cols = plt.cm.RdYlGn(np.array(esgs)/100)
    bars = ax.barh(lbls, vals, color=cols, edgecolor='white', linewidth=1.5)
    for bar, e in zip(bars, esgs):
        ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                f'ESG {e}', va='center', fontsize=8.5)
    r,v = port_stats(w, rets_used)
    ax.set_xlabel('Poids (%)')
    ax.set_title(f'{titre}\nSharpe={(r-RF)/v:.3f} | ESG={w@esg_used:.1f}', fontweight='bold')
    ax.set_xlim(0, max(vals)*1.25 if vals else 10)

plt.suptitle('Évolution de la composition — Acte 1 → Acte 2', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n📢 Questions pour le pitch :')
print('  1. Quels actifs avez-vous renforcés/réduits entre Acte 1 et Acte 2 ? Pourquoi ?')
print('  2. Quel est votre coût ESG (perte de Sharpe) ? Est-ce justifié ?')
print('  3. Si le seuil ESG passait à 70, que feriez-vous ?')

---
# 📊 CORRECTION — Le portefeuille optimal & synthèse

> **Professeur : exécutez cette section après tous les pitchs.**

In [ ]:
print('='*60)
print('  CORRECTION — Portefeuille optimal (Max Sharpe, ESG ≥ 55)')
print('='*60)

r_e, v_e = port_stats(w_opt_e2, RETS_CHOC)
print(f'\n  Rendement : {r_e*100:.2f}% | Volatilité : {v_e*100:.2f}%')
print(f'  Sharpe    : {(r_e-RF)/v_e:.3f} | ESG moyen : {w_opt_e2@ESG_CHOC:.1f}')
print()

df_opt = pd.DataFrame({
    'Actif':   NAMES,
    'Poids optimal (%)': (w_opt_e2*100).round(1),
    'Poids vos Acte2 (%)': (w2*100).round(1),
    'ESG': ESG_CHOC,
})
df_opt = df_opt[(df_opt['Poids optimal (%)']>0.5)|(df_opt['Poids vos Acte2 (%)']>0.5)]
df_opt['Écart (pp)'] = (df_opt['Poids vos Acte2 (%)'] - df_opt['Poids optimal (%)']).round(1)
display(df_opt.style.applymap(color_esg, subset=['ESG']))

# Leçons clés
print('\n📚 LEÇONS CLÉS')
print('─'*50)
r_libre,v_libre = port_stats(w_opt_c2, RETS_CHOC)
cout_esg = ((r_libre-RF)/v_libre - (r_e-RF)/v_e) / ((r_libre-RF)/v_libre) * 100
print(f'  1. Coût ESG sur Sharpe     : {cout_esg:.1f}%')
print(f'     → Faible / élevé selon votre jugement')
print(f'  2. ESG moyen optimal       : {w_opt_e2@ESG_CHOC:.1f} (seuil imposé : {ESG_MIN})')
print(f'     → Satisfaire la contrainte ≠ maximiser l\'ESG')
print(f'  3. Événement : impact sur rendement ajusté')
print(f'     → Diversification protège, concentration amplifie')
print()
print('  📖 Références :')
print('     Markowitz (1952) · Pedersen et al. (2021, JFE)')
print('     Règlement SFDR (UE) 2019/2088')

---
## 🏆 Classement final des équipes

In [ ]:
# Le professeur peut saisir les scores des équipes ici
# ══════════════════════════════════════════════════════
scores_equipes = {
    'MonEquipe':  score_total,  # calculé automatiquement
    # Ajouter les autres équipes manuellement :
    # 'Équipe A': 95.0,
    # 'Équipe B': 88.5,
    # 'Équipe C': 102.0,
}
# ══════════════════════════════════════════════════════

df_lb = pd.DataFrame([
    {'Rang': i+1, 'Équipe': t, 'Score /160': s,
     'Perf. (%)': f'{s/160*100:.0f}%'}
    for i,(t,s) in enumerate(sorted(scores_equipes.items(), key=lambda x:-x[1]))
])

fig, ax = plt.subplots(figsize=(9, 1.2 + len(df_lb)*0.55))
ax.axis('off')
tbl = ax.table(cellText=df_lb.values, colLabels=df_lb.columns,
               cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(12)
podium = {0:'#FFD700', 1:'#C0C0C0', 2:'#CD7F32'}
for j in range(len(df_lb.columns)):
    tbl[0,j].set_facecolor(C_BLUE)
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(len(df_lb)):
    c = podium.get(i, '#F8F9FA')
    for j in range(len(df_lb.columns)): tbl[i+1,j].set_facecolor(c)
ax.set_title('🏆 GreenAlpha Challenge — Classement Final',
             fontsize=14, fontweight='bold', color=C_BLUE, pad=15)
plt.tight_layout(); plt.show()